In [ ]:
!pip install "psycopg[binary]==3.3.6" "python-dotenv==1.2.4"


In [ ]:
import os
import psycopg
from psycopg import sql
from dotenv import load_dotenv

load_dotenv("../../.env")
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD") or None
PGDATABASE = os.getenv("PGDATABASE", "university")
PGSSLMODE = os.getenv("PGSSLMODE", "prefer")
conn_args = dict(host=PGHOST, port=PGPORT, user=PGUSER, password=PGPASSWORD, sslmode=PGSSLMODE)

with psycopg.connect(autocommit=True, **{**conn_args, "dbname": "postgres"}) as admin:
    if not admin.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PGDATABASE,)).fetchone():
        admin.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(PGDATABASE)))
conn = psycopg.connect(autocommit=True, dbname=PGDATABASE, **conn_args)
print("connected")


In [ ]:
def show(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description is None:
            print(cur.statusmessage); return
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    cells = [["NULL" if v is None else str(v) for v in row] for row in rows]
    widths = [max([len(c)] + [len(row[i]) for row in cells]) for i,c in enumerate(cols)]
    print(" | ".join(c.center(w) for c,w in zip(cols,widths)).rstrip())
    print("-+-".join("-"*w for w in widths))
    for row in cells:
        print(" | ".join(v.ljust(w) for v,w in zip(row,widths)).rstrip())
    print(f"({len(rows)} row{'' if len(rows)==1 else 's'})")


In [ ]:
conn.execute("REVOKE ALL ON ALL TABLES IN SCHEMA public FROM app_reader, app_writer")
conn.execute("REVOKE USAGE ON SCHEMA public FROM app_reader, app_writer")
conn.execute("DROP TABLE IF EXISTS patients CASCADE")
conn.execute("DROP ROLE IF EXISTS app_reader, app_writer")
conn.close()
print("done")


- **What happens:** create roles, grant schema and table privileges (least privilege).
- **Why:** control who can access what.


In [ ]:
conn.execute("SET ROLE app_reader")
show("SELECT id, name, clinic_id FROM patients ORDER BY id")
conn.execute("RESET ROLE")


- **What happens:** switch to role without logging in; see if SELECT works.
- **Why:** `SET ROLE` is great for testing policies/privileges.


In [ ]:
conn.execute("ALTER TABLE patients ENABLE ROW LEVEL SECURITY")
conn.execute("DROP POLICY IF EXISTS patients_clinic ON patients")
conn.execute("CREATE POLICY patients_clinic ON patients FOR SELECT TO app_reader USING (clinic_id = 1)")
conn.execute("GRANT SELECT ON patients TO app_reader")
conn.execute("SET ROLE app_reader")
show("SELECT id, name, clinic_id FROM patients ORDER BY id")
conn.execute("RESET ROLE")


- **What happens:** RLS filters rows so reader only sees clinic_id=1.
- **Why:** enforce data isolation per tenant/clinic.

```mermaid
sequenceDiagram
  participant A as app_reader
  participant DB as Postgres
  A->>DB: SELECT * FROM patients
  DB->>DB: Apply RLS policy (clinic_id=1)
  DB-->>A: Only alice,bob
```


In [ ]:
show("SHOW password_encryption")
show("SHOW ssl")
show("SELECT * FROM pg_hba_file_rules LIMIT 5")


- **What happens:** inspect password encryption, SSL status, and pg_hba rules.
- **Why:** know auth/TLS configuration; md5 deprecated, prefer scram-sha-256.


In [ ]:
conn.execute("CREATE EXTENSION IF NOT EXISTS pgcrypto")
show("SELECT encode(digest('hello','sha256'),'hex') AS sha256_hex")
show("SELECT crypt('s3cr3t', gen_salt('bf')) LIKE '$2%' AS is_bcrypt")
show("SELECT crypt('s3cr3t', crypt('s3cr3t', gen_salt('bf'))) = crypt('s3cr3t', crypt('s3cr3t', gen_salt('bf'))) AS match_demo")


- **What happens:** use `digest` for checksums and `crypt`/`gen_salt` for password hashing.
- **Why:** use tested crypto, salt passwords, never store plaintext.


In [ ]:
with conn.cursor() as cur:
    cur.execute("PREPARE get_patient(int) AS SELECT id, name, clinic_id FROM patients WHERE id = $1")
    cur.execute("EXECUTE get_patient(1)")
    print("prepared result:", cur.fetchall())
    cur.execute("DEALLOCATE get_patient")


- **What happens:** prepare a parameterized statement and execute with values.
- **Why:** prevents SQL injection and can be reused; always pass values as parameters, not string concatenation.


In [ ]:
show("SELECT name, setting FROM pg_settings WHERE name IN ('log_statement','log_connections','log_disconnections') ORDER BY name")


- **What happens:** check common logging settings.
- **Why:** enable appropriate logging for auditing in production (consider pgaudit extension).


In [ ]:
conn.execute("DROP TABLE IF EXISTS patients CASCADE")
conn.execute("DROP ROLE IF EXISTS app_reader, app_writer")
conn.close()
print("done")
